# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q textblob
    print('Packages installed.')

    # Download NLTK data
    import nltk
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)
    nltk.download('punkt_tab', quiet=True)
    print('NLTK data downloaded.')


# Sentiment Analysis In-Depth

This notebook provides a deep dive into sentiment analysis using two popular libraries: TextBlob and NLTK (VADER). We will compare their performance on a dataset of reviews, clean the text, and visualize the results.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk
import re
from wordcloud import WordCloud

# Ensure VADER lexicon is downloaded
nltk.download('vader_lexicon', quiet=True)
nltk.download('stopwords', quiet=True)

sns.set_style('whitegrid')

## 1. Load and Prepare Data
We will use a comprehensive set of dummy movie reviews to simulate a real-world scenario.

In [ ]:
reviews = [
    "I absolutely loved this movie! The acting was superb and the plot was engaging.",
    "What a waste of time. The story made no sense and the characters were flat.",
    "It was okay. Not the best I've seen, but entertaining enough for a Friday night.",
    "A masterpiece of modern cinema. Visually stunning.",
    "I fell asleep halfway through. Boring and repetitive.",
    "Great soundtrack, but the dialogue was a bit cheesy.",
    "Terrible direction. The camera work gave me a headache.",
    "An emotional rollercoaster. I cried at the end.",
    "The special effects were good, but the acting was wooden.",
    "I would watch it again. Highly recommended!",
    "Avoid at all costs. It is not worth your money.",
    "Surprisingly good given the low budget.",
    "The villain was fantastic, but the hero was annoying.",
    "Pure garbage. I walked out of the theater.",
    "A decent attempt, but it falls short of the original.",
    "Hilarious! I laughed so hard my sides hurt.",
    "Depressing and slow. Not my cup of tea.",
    "Brilliant performance by the lead actor.",
    "Confusing plot twists that didn't pay off.",
    "A fun, action-packed ride from start to finish."
]

df = pd.DataFrame(reviews, columns=['text'])
print(f"Loaded {len(df)} reviews.")
print(df.head())

## 2. Text Preprocessing
Before analysis, it's good practice to clean the text. We will lowercase everything and remove special characters.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove punctuation/numbers
    return text

df['clean_text'] = df['text'].apply(clean_text)
print(df[['text', 'clean_text']].head())

## 3. TextBlob Analysis
TextBlob calculates polarity (-1 to 1) and subjectivity (0 to 1). Polarity measures how positive or negative the text is, while subjectivity measures how much of it is opinion vs fact.

In [ ]:
def get_textblob_sentiment(text):
    blob = TextBlob(text)
    return blob.sentiment.polarity, blob.sentiment.subjectivity

df[['tb_polarity', 'tb_subjectivity']] = df['clean_text'].apply(
    lambda x: pd.Series(get_textblob_sentiment(x))
)
print(df[['text', 'tb_polarity', 'tb_subjectivity']].head())

## 4. VADER Analysis
VADER (Valence Aware Dictionary and sEntiment Reasoner) is rule-based and specifically tuned for social media sentiment. It handles emojis, capitalization, and intensifiers better than TextBlob, though our cleaning removed some of those cues. Let's see how it performs on the raw text.

In [ ]:
sia = SentimentIntensityAnalyzer()

def get_vader_sentiment(text):
    # Note: VADER works best on raw text (with punctuation/caps)
    return sia.polarity_scores(text)['compound']

df['vader_compound'] = df['text'].apply(get_vader_sentiment)
print(df[['text', 'tb_polarity', 'vader_compound']].head())

## 5. Comparative Visualization
Let's compare the two methods. Do they agree?

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='tb_polarity', y='vader_compound', s=100, alpha=0.7)
plt.title('TextBlob Polarity vs VADER Compound Score')
plt.xlabel('TextBlob Polarity')
plt.ylabel('VADER Compound Score')
plt.axhline(0, color='gray', linestyle='--')
plt.axvline(0, color='gray', linestyle='--')
plt.show()

## 6. Word Clouds
Visualizing the most frequent words in positive vs negative reviews (based on VADER).

In [ ]:
positive_text = ' '.join(df[df['vader_compound'] > 0.05]['clean_text'])
negative_text = ' '.join(df[df['vader_compound'] < -0.05]['clean_text'])

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

if positive_text:
    wc_pos = WordCloud(background_color='white', colormap='Greens').generate(positive_text)
    axes[0].imshow(wc_pos, interpolation='bilinear')
    axes[0].set_title('Positive Words')
    axes[0].axis('off')

if negative_text:
    wc_neg = WordCloud(background_color='white', colormap='Reds').generate(negative_text)
    axes[1].imshow(wc_neg, interpolation='bilinear')
    axes[1].set_title('Negative Words')
    axes[1].axis('off')

plt.show()

## Conclusion
By comparing TextBlob and VADER, researchers can choose the tool that best fits their specific data type (e.g., formal text vs. social media). VADER is generally more robust for short, informal text containing emojis and slang.